In [1]:
!pip install -q transformers accelerate peft bitsandbytes datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 14.2 MB/s eta 0:00:00


In [2]:
import torch, os
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model
from datasets import load_dataset

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
MODEL_NAME = "Qwen/Qwen3-4B"


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
print("✅ Model loaded with LoRA + 4-bit quantization.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/3.99G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

✅ Model loaded with LoRA + 4-bit quantization.


In [4]:
dataset = load_dataset("json", data_files="Finetuning_data.json")

def format_and_tokenize(batch):
    text = f"### Instruction:\n{batch['instruction']}\n\n### Response:\n{batch['response']}"
    tokens = tokenizer(text, truncation=True, padding="max_length", max_length=512)
    return tokens

train_data = dataset["train"].map(format_and_tokenize, remove_columns=dataset["train"].column_names)
print("✅ Dataset ready:", len(train_data), "samples")


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/678 [00:00<?, ? examples/s]

✅ Dataset ready: 678 samples


In [5]:
training_args = TrainingArguments(
    output_dir="./qwen3_finetuned",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=True,
    optim="paged_adamw_32bit",
    save_steps=100,
    logging_steps=20,
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)
trainer = Trainer(
    model=model,
    train_dataset=train_data,
    args=training_args,
    data_collator=data_collator
)

torch.cuda.empty_cache()
trainer.train()


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
20,2.519100
40,2.098100
60,1.940300
80,1.912900
100,1.880300
120,1.843100
140,1.787200
160,1.809700


TrainOutput(global_step=170, training_loss=1.9624250187593348, metrics={'train_runtime': 2118.9605, 'train_samples_per_second': 0.64, 'train_steps_per_second': 0.08, 'total_flos': 1.5148158541627392e+16, 'train_loss': 1.9624250187593348, 'epoch': 2.0})

In [6]:
OUTPUT_DIR = "./qwen3_lora_adapter"
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Finetuning complete — adapter saved at {OUTPUT_DIR}")


✅ Finetuning complete — adapter saved at ./qwen3_lora_adapter


In [7]:
from google.colab import files
!zip -r qwen3_lora_adapter.zip ./qwen3_lora_adapter
files.download("qwen3_lora_adapter.zip")

  adding: qwen3_lora_adapter/ (stored 0%)
  adding: qwen3_lora_adapter/tokenizer.json (deflated 81%)
  adding: qwen3_lora_adapter/README.md (deflated 65%)
  adding: qwen3_lora_adapter/special_tokens_map.json (deflated 69%)
  adding: qwen3_lora_adapter/vocab.json (deflated 61%)
  adding: qwen3_lora_adapter/chat_template.jinja (deflated 76%)
  adding: qwen3_lora_adapter/merges.txt (deflated 57%)
  adding: qwen3_lora_adapter/tokenizer_config.json (deflated 90%)
  adding: qwen3_lora_adapter/adapter_model.safetensors (deflated 8%)
  adding: qwen3_lora_adapter/adapter_config.json (deflated 56%)
  adding: qwen3_lora_adapter/added_tokens.json (deflated 68%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
import time
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM
from tqdm.auto import tqdm

BASE_MODEL = "Qwen/Qwen3-4B"
ADAPTER_PATH = "./qwen3_lora_adapter"
OUTPUT_PATH = "./qwen3_finetuned_full"

print("🔄 Loading base model...")
start_time = time.time()

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

load_time = time.time() - start_time
print(f"✅ Base Model Loaded in {load_time:.2f} seconds\n")

print("🔗 Attaching LoRA adapter...")
start_time = time.time()

model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

attach_time = time.time() - start_time
print(f"✅ LoRA Adapter Loaded in {attach_time:.2f} seconds\n")

print("🧩 Merging LoRA weights into base model (this may take 2–5 minutes)...")

start_time = time.time()
# Fake progress bar for user feedback
for _ in tqdm(range(100), desc="Merging LoRA", unit="%"):
    time.sleep(0.03)  # cosmetic delay for progress bar

merged_model = model.merge_and_unload()
merge_time = time.time() - start_time

print(f"\n🎉 Merge Complete in {merge_time:.2f} seconds")
print("💾 Saving final merged model...\n")

start_time = time.time()
merged_model.save_pretrained(OUTPUT_PATH)
save_time = time.time() - start_time

print(f"✅ Model saved at: {OUTPUT_PATH}")
print(f"🕒 Save time: {save_time:.2f} seconds")



🔄 Loading base model...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

✅ Base Model Loaded in 32.27 seconds

🔗 Attaching LoRA adapter...
✅ LoRA Adapter Loaded in 0.18 seconds

🧩 Merging LoRA weights into base model (this may take 2–5 minutes)...


Merging LoRA:   0%|          | 0/100 [00:00<?, ?%/s]


🎉 Merge Complete in 3.09 seconds
💾 Saving final merged model...

✅ Model saved at: ./qwen3_finetuned_full
🕒 Save time: 454.45 seconds


In [10]:
from google.colab import files

print("📦 Zipping model...")
!zip -r qwen3_finetuned_full.zip ./qwen3_finetuned_full

print("⬇️ Downloading .zip...")
files.download("qwen3_finetuned_full.zip")

print("🎉 Your merged model is ready for LM Studio!")


📦 Zipping model...
updating: qwen3_finetuned_full/ (stored 0%)
updating: qwen3_finetuned_full/model-00001-of-00002.safetensors (deflated 21%)
updating: qwen3_finetuned_full/generation_config.json (deflated 37%)
updating: qwen3_finetuned_full/config.json (deflated 74%)
updating: qwen3_finetuned_full/model-00002-of-00002.safetensors (deflated 20%)
updating: qwen3_finetuned_full/model.safetensors.index.json (deflated 96%)
⬇️ Downloading .zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🎉 Your merged model is ready for LM Studio!
